---
title: "DRG Merge"

author: "Carlos Resurreccion"

date: "2024-08-12"

---

In [1]:
source(here::here("data-cleaning/r_scripts", "00_libraries-params.R"))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc_gmail_com/drg-pipeline

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: docstring


Attaching package: ‘docstring’


The following object is masked from ‘package:utils’:

    ?


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The foll

In [2]:
# File Path Prefixes:
clean_prefix <- "data-cleaning"
data_prefix <- file.path(clean_prefix, "data")
chkpt_path <- file.path(data_prefix, "checkpoints")
checkpoint_2_path <- file.path(chkpt_path, "checkpoint_2_master_clean_claims")
checkpoint_5_path <- file.path(chkpt_path, "checkpoint_5_thai_output")
checkpoint_6_path <- file.path(chkpt_path, "checkpoint_6_thai_merged")
claims_fpath <- here(checkpoint_2_path, "checkpoint_2_claims_2018_sampled_6282_.csv")
grouper_fpath <- here(checkpoint_5_path, "CHECKPOINT_4_THAI_GROUPER_INPUT_2018_SAMPLED_6282_Res.TXT")
merged_fpath <- here(checkpoint_6_path, "checkpoint_6_grouped_claims.csv")
gcp_proj <- "drg-pipeline"
bq_dataset <- "phic"
bq_table <- "temp_claims_new"


In [3]:
claims <- fread(claims_fpath, colClasses = "character")
claims[, caseid := 1:.N]
grouper <- fread(grouper_fpath, sep = "|", na.strings = "--", header = TRUE)
result <- merge(claims, grouper, by = "caseid", all.x = TRUE)
result[, drgname := NULL]


In [4]:
# Define the schema using bq_field
table_schema <- list(
  bq_field("caseid", "INT64"),
  bq_field("id_series", "STRING"),
  bq_field("id_pin", "STRING"),
  bq_field("date_adm", "DATE"),
  bq_field("time_adm", "TIME"),
  bq_field("date_dis", "DATE"),
  bq_field("time_dis", "TIME"),
  bq_field("date_rec", "DATE"),
  bq_field("date_ref", "DATE"),
  bq_field("date_check", "DATE"),
  bq_field("id_hci", "STRING"),
  bq_field("id_hcp", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("clin_outpatient", "BOOL"),
  bq_field("clin_emergency", "BOOL"),
  bq_field("pat_type", "STRING"),
  bq_field("clin_acc", "STRING"),
  bq_field("pat_rel", "STRING"),
  bq_field("pat_bdate", "DATE"),
  bq_field("pat_age", "FLOAT64"),
  bq_field("pat_sex", "STRING"),
  bq_field("pat_bwt", "FLOAT64"),
  bq_field("pat_memcat_parent", "STRING"),
  bq_field("pat_memcat_child", "STRING"),
  bq_field("clin_discharge", "INT64"),
  bq_field("clin_c1", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("clin_c2", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("claim_status", "STRING"),
  bq_field("claim_payout", "FLOAT64"),
  bq_field("claim_charge", "FLOAT64"),
  bq_field("date_ext", "DATE"),
  bq_field("id_year", "INT64"),
  bq_field("clin_icd", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("clin_rvs", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("clin_c1_orig", "STRING"),
  bq_field("clin_c2_orig", "STRING"),
  bq_field("icd9_list", "STRING", mode = "REPEATED"), # Array of strings
  bq_field("pdx", "STRING"),
  bq_field("pdx_code", "INT64"),
  bq_field("drg", "INT64"),
  bq_field("rw", "FLOAT64"),
  bq_field("wtlos", "FLOAT64"),
  bq_field("ot", "INT64"),
  bq_field("adjrw", "FLOAT64"),
  bq_field("err", "INT64"),
  bq_field("warn", "INT64"),
  bq_field("los", "INT64")
)

bq_table_exists <- function(gcp_proj, bq_dataset, bq_table) {
  tryCatch(
    {
      bq_table(gcp_proj, bq_dataset, bq_table)
      TRUE # Table exists
    },
    error = function(e) {
      FALSE # Table does not exist
    }
  )
}

if (!bq_table_exists(gcp_proj, bq_dataset, bq_table)) {
  bq_table_create(
    bq_table(gcp_proj, bq_dataset, bq_table),
    fields = table_schema
  )
}


In [5]:
# Convert data types to match BigQuery schema
result[, caseid := as.integer(caseid)]
result[, id_series := as.character(id_series)]
result[, id_pin := as.character(id_pin)]
result[, date_adm := as.Date(date_adm, format = "%m/%d/%Y")]
result[, time_adm := as.ITime(time_adm)]
result[, date_dis := as.Date(date_dis, format = "%m/%d/%Y")]
result[, time_dis := as.ITime(time_dis)]
result[, date_rec := as.Date(date_rec, format = "%m/%d/%Y")]
result[, date_ref := as.Date(date_ref, format = "%m/%d/%Y")]
result[, date_check := as.Date(date_check, format = "%m/%d/%Y")]
result[, id_hci := as.character(id_hci)]

# Convert character "0"/"1" to logical for Boolean fields
result[, clin_outpatient := as.logical(as.integer(clin_outpatient))]
result[, clin_emergency := as.logical(as.integer(clin_emergency))]

result[, pat_type := as.character(pat_type)]
result[, clin_acc := as.character(clin_acc)]
result[, pat_rel := as.character(pat_rel)]
result[, pat_bdate := as.Date(pat_bdate, format = "%m/%d/%Y")]
result[, pat_age := as.numeric(pat_age)]
result[, pat_sex := as.character(pat_sex)]
result[, pat_bwt := as.numeric(pat_bwt)]
result[, pat_memcat_parent := as.character(pat_memcat_parent)]
result[, pat_memcat_child := as.character(pat_memcat_child)]
result[, clin_discharge := as.integer(clin_discharge)]

result[, claim_status := as.character(claim_status)]
result[, claim_payout := as.numeric(claim_payout)]
result[, claim_charge := as.numeric(claim_charge)]
result[, date_ext := as.Date(date_ext, format = "%m/%d/%Y")]
result[, id_year := as.integer(id_year)]

result[, clin_c1_orig := as.character(clin_c1_orig)]
result[, clin_c2_orig := as.character(clin_c2_orig)]

result[, pdx := as.character(pdx)]
result[, pdx_code := as.integer(pdx_code)]
result[, drg := as.integer(drg)]
result[, rw := as.numeric(rw)]
result[, wtlos := as.numeric(wtlos)]
result[, ot := as.integer(ot)]
result[, adjrw := as.numeric(adjrw)]
result[, err := as.integer(err)]
result[, warn := as.integer(warn)]
result[, los := as.integer(los)]

fwrite(result, merged_fpath)
# Convert to list of strings for repeated fields (arrays)
result[, id_hcp := strsplit(id_hcp, "\\|\\|")]
# Convert to list of strings for repeated fields (arrays)
result[, clin_c1 := strsplit(clin_c1, "\\|\\|")]
result[, clin_c2 := strsplit(clin_c2, "\\|\\|")]
# Convert to list of strings for repeated fields (arrays)
result[, clin_icd := strsplit(clin_icd, "\\|\\|")]
result[, clin_rvs := strsplit(clin_rvs, "\\|\\|")]
# Convert to list of strings for repeated fields (arrays)
result[, icd9_list := strsplit(icd9_list, "\\|\\|")]


In [6]:
# Upload the data.table to BigQuery
bq_table_upload(
  bq_table(gcp_proj, bq_dataset, bq_table),
  values = result,
  write_disposition = "WRITE_TRUNCATE" # Options: WRITE_TRUNCATE, WRITE_APPEND, WRITE_EMPTY
)
